# Extended Data Figure 4 — Average effect size for eQTLs and cis-pQTLs

Mean absolute rescaled effect size (|β̂_rescaled|) ± 95 % CI across five variant
consequence categories for eQTLs and cis-pQTLs.

The five consequence categories (ranked by severity):

1. protein_altering
2. promoter
3. enhancer
4. intragenic
5. intergenic

**Source notebook:** `chapters/02-analysis/02-variant-effects/03_variant_regulatory_consequence.ipynb`  
**Data:** `data/intermediate_files/lead_variant_consequence_exploded/`
(pre-computed by the source notebook; re-run that notebook if missing)


## Setup


In [1]:
from __future__ import annotations

import plotnine as p9
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import OpenTargetsTheme
from manuscript_methods.consequence import ConsequenceCategory
from manuscript_methods.study_statistics import StudyType

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 15:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Paths


In [3]:
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

# Pre-computed by 03_variant_regulatory_consequence.ipynb
consequence_dataset_path = path_to_intermediate_data_folder + "lead_variant_consequence_exploded"

## Load consequence dataset

This was pre-computed by `03_variant_regulatory_consequence.ipynb` and saved to parquet.


In [4]:
consequence_dataset = session.spark.read.parquet(consequence_dataset_path)
print(f"Consequence dataset: {consequence_dataset.count():,} rows")
consequence_dataset.printSchema()

Consequence dataset: 261,334 rows
root
 |-- studyType: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- variantId: string (nullable = true)
 |-- maxAbsEstimatedBeta: double (nullable = true)
 |-- distanceFromTss: long (nullable = true)
 |-- consequenceCategory: string (nullable = true)
 |-- consequenceSource: string (nullable = true)
 |-- partition: string (nullable = true)



## Rank consequences per variant × study type

For each (variantId, studyType, partition) triplet, assign a severity ranking and keep the most severe consequence.


In [5]:
w_rank = Window.partitionBy("variantId", "studyType", "partition").orderBy(f.asc("ranking"))

consequence_dataset_ranked = (
    consequence_dataset.withColumn("ranking", ConsequenceCategory.ranking(f.col("consequenceCategory")))
    .withColumn("lowestInRanking", f.dense_rank().over(w_rank))
    .withColumn("maxAbsEstimatedBeta", f.max("maxAbsEstimatedBeta").over(w_rank))
    .withColumn("con", f.size(f.collect_list("consequenceCategory").over(w_rank)))
    .dropDuplicates(["variantId", "consequenceCategory", "partition"])
    .orderBy("variantId")
)
print(f"Ranked dataset: {consequence_dataset_ranked.count():,} rows")

Ranked dataset: 173,195 rows


## Extended Data Figure 4

Filter to **eQTL** and **cis-pQTL** studies, compute mean |β| per consequence category with 95 % CI, then plot.


In [6]:
# Filter to molecular QTL study types
w6 = Window.partitionBy("studyType", "consequenceCategory")
molqtl_studies = [StudyType.EQTL.value, StudyType.CIS_PQTL.value]

molqtl_consequence_vs_beta = (
    consequence_dataset_ranked.filter(f.col("studyType").isin(*molqtl_studies))
    .withColumn("nConsequence", f.count("variantId").over(w6))
    .withColumn("avgMaxAbsEstimatedBeta", f.avg("maxAbsEstimatedBeta").over(w6))
    .withColumn("stdMaxAbsEstimatedBeta", f.stddev("maxAbsEstimatedBeta").over(w6))
    .withColumn("seMaxAbsEstimatedBeta", f.col("stdMaxAbsEstimatedBeta") / f.sqrt(f.col("nConsequence")))
    .withColumn("CILower", f.col("avgMaxAbsEstimatedBeta") - 1.96 * f.col("seMaxAbsEstimatedBeta"))
    .withColumn("CIUpper", f.col("avgMaxAbsEstimatedBeta") + 1.96 * f.col("seMaxAbsEstimatedBeta"))
    .drop("lowestInRanking", "variantId", "maxAbsEstimatedBeta")
    .drop_duplicates(["studyType", "consequenceCategory"])
    .cache()
)
print(f"molQTL consequence vs beta: {molqtl_consequence_vs_beta.count():,}")
molqtl_consequence_vs_beta.toPandas().head()

26/04/15 15:38:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


molQTL consequence vs beta: 10


,studyType,geneId,distanceFromTss,consequenceCategory,consequenceSource,partition,ranking,con,nConsequence,avgMaxAbsEstimatedBeta,stdMaxAbsEstimatedBeta,seMaxAbsEstimatedBeta,CILower,CIUpper
0,eqtl,ENSG00000095485,-2112,enhancer,interval,"ENSG00000196072,eqtl",3,1,13050,1.000915,0.466233,0.004081,0.992916,1.008915
1,cis-pqtl,ENSG00000138175,-107,intragenic,vep,"ENSG00000138175,cis-pqtl",4,3,787,0.370050,0.282863,0.010083,0.350288,0.389813
2,cis-pqtl,ENSG00000305032,82180,intergenic,vep,"ENSG00000107554,cis-pqtl",6,2,481,0.392021,0.288804,0.013168,0.366211,0.417831
3,cis-pqtl,ENSG00000121898,22224,enhancer,interval,"ENSG00000121898,cis-pqtl",3,1,251,0.353456,0.259932,0.016407,0.321299,0.385613
4,eqtl,ENSG00000120055,-565,protein_altering,vep,"ENSG00000269609,eqtl",1,1,1816,1.159448,0.566094,0.013284,1.133411,1.185484


In [7]:
fig_ed4 = (
    molqtl_consequence_vs_beta.toPandas()
    >> p9.ggplot()
    + p9.geom_point(
        p9.aes(x="consequenceCategory", y="avgMaxAbsEstimatedBeta", color="studyType"),
        size=1.5,
        position=p9.position_dodge(width=0.3),
    )
    + p9.geom_errorbar(
        p9.aes(x="consequenceCategory", ymin="CILower", ymax="CIUpper", color="studyType"),
        width=0.3,
        position=p9.position_dodge(width=0.3),
    )
    + p9.scale_color_manual(
        values={"eqtl": "#2ca02c", "cis-pqtl": "#9467bd"},
        labels={"eqtl": "eQTL", "cis-pqtl": "cis-pQTL"},
        name="Study type",
    )
    + OpenTargetsTheme.theme
    + p9.theme(axis_text_y=p9.element_text(rotation=0))
    + p9.labs(
        x="",
        y=r"$|\hat{\beta}_{\mathrm{rescaled}}|$",
    )
    + p9.theme(legend_position="bottom", figure_size=(5, 4))
    + p9.geom_hline(p9.aes(yintercept=0), linetype="dashed", color="red", size=0.5)
    + p9.coord_flip()
)
fig_ed4
fig_ed4.save("extended_figure_4.pdf", dpi=300)

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning:

Saving 5 x 4 in image.

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning:

Filename: extended_figure_4.pdf

